# Ensembles más allá de los árboles

Los [bosques aleatorios](../random_forests.ipynb) promedian árboles, y el [gradient boosting](../gradient_boosting.ipynb) los suma en secuencia. Ninguna de las dos ideas se limita a los árboles: cualquier modelo se puede promediar, combinar con un modelo entrenado para ello, o someter a *bagging*. Aquí se prueban las tres con las viviendas de California, medidas con las filas de validación de [Validación y selección de modelos](../../../ml-fundamentals/validation_and_model_selection.ipynb), para ver cuándo ayuda combinar, y cuándo no.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = 100_000 * housing["MedHouseVal"]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_validation, y_train, y_validation = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)


def validation_mae(model):
    return round(mean_absolute_error(y_validation, model.fit(X_train, y_train).predict(X_validation)))

## Tres modelos distintos

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

members = {
    "linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "k-nearest neighbours, distance weights": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10, weights="distance")),
    "gradient boosting": HistGradientBoostingRegressor(random_state=42),
}
{name: validation_mae(model) for name, model in members.items()}

{'linear regression': 53333,
 'k-nearest neighbours, distance weights': 43924,
 'gradient boosting': 32175}

## Votación: la media de las predicciones

`VotingRegressor` entrena cada modelo y predice la media de sus predicciones (`VotingClassifier` toma el voto mayoritario, o promedia las probabilidades):

In [ ]:
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

{
    "the three models": validation_mae(VotingRegressor(list(members.items()))),
    "boosting and a random forest": validation_mae(VotingRegressor([
        ("gradient boosting", HistGradientBoostingRegressor(random_state=42)),
        ("random forest", RandomForestRegressor(random_state=42, n_jobs=-1)),
    ])),
}

{'the three models': 39421, 'boosting and a random forest': 31960}

Promediar los tres da unos 39 400 \$, peor que el gradient boosting solo (32 200 \$): cada predicción se ve arrastrada hacia los errores mayores del modelo lineal y de los vecinos. Promediar el boosting con un bosque aleatorio, dos modelos fuertes construidos de formas distintas, da unos 32 000 \$, un poco mejor que cualquiera de los dos (el bosque solo puntuó unos 33 600 \$ en [Validación y selección de modelos](../../../ml-fundamentals/validation_and_model_selection.ipynb)). La ganancia es pequeña, y aquí solo vino de miembros que ya eran buenos.

## Stacking: un modelo que aprende a combinar

En lugar de una media simple, el **stacking** entrena un segundo modelo, el *estimador final*, para combinar las predicciones de los miembros. Para entrenarlo con predicciones que los miembros no hicieron sobre sus propias filas de entrenamiento, `StackingRegressor` usa predicciones de validación cruzada (`cv=5`); después, una regresión Ridge aprende un peso por miembro:

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

stacking = StackingRegressor(list(members.items()), final_estimator=RidgeCV(), cv=5)
validation_mae(stacking), dict(zip(members, stacking.final_estimator_.coef_.round(2)))

(32133,
 {'linear regression': np.float64(-0.0),
  'k-nearest neighbours, distance weights': np.float64(0.01),
  'gradient boosting': np.float64(1.0)})

El estimador final dio al gradient boosting un peso de 1,0 y a los otros dos casi nada: aprendió, a partir de predicciones de validación cruzada, que la mejor forma de usar los miembros débiles es ignorarlos. El resultado, unos 32 100 \$, es prácticamente el del propio boosting (32 200 \$). El stacking solo vuelve a ponderar lo que predicen los miembros: con un miembro fuerte y dos débiles, la mejor ponderación se queda con el fuerte, y evita la pérdida que sufrió la votación simple. Con miembros fuertes que cometen errores distintos, puede superar a cada uno de ellos.

## Bagging de cualquier modelo

El [bagging](../random_forests.ipynb#muchos-árboles-con-datos-remuestreados) entrena cada copia de un modelo con una muestra *bootstrap* y las promedia; `BaggingRegressor` lo hace para cualquier modelo. Con un solo árbol de decisión, y con k vecinos más cercanos:

In [ ]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

neighbours = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10))
{
    "one tree": validation_mae(DecisionTreeRegressor(random_state=42)),
    "100 bagged trees": validation_mae(BaggingRegressor(DecisionTreeRegressor(random_state=42), n_estimators=100,
                                                        random_state=42, n_jobs=-1)),
    "k-nearest neighbours, uniform weights": validation_mae(neighbours),
    "30 bagged k-nearest neighbours": validation_mae(BaggingRegressor(neighbours, n_estimators=30, random_state=42, n_jobs=-1)),
}

{'one tree': 45594,
 '100 bagged trees': 33497,
 'k-nearest neighbours, uniform weights': 44462,
 '30 bagged k-nearest neighbours': 43622}

El bagging convierte los 45 600 \$ de un árbol en 33 500 \$, una cuarta parte menos: un solo árbol profundo cambia mucho con sus filas de entrenamiento, y promediar copias *bootstrap* anula buena parte de eso. El mismo bagging apenas mueve k vecinos más cercanos, de 44 500 \$ a 43 600 \$: sus predicciones ya son medias de diez vecinos, así que remuestrear las filas las cambia poco. El bagging es un remedio contra la varianza, y solo ayuda donde hay varianza que remediar. Con los valores por defecto de scikit-learn para regresión, cien árboles con bagging *son* un bosque aleatorio: la elección aleatoria de variables en cada división solo empieza cuando `max_features` es menor que 1.

## Cuándo ayuda combinar

Un ensemble mejora a sus miembros cuando estos son **buenos** y cometen errores **distintos**. Promediar un modelo fuerte con otros débiles lo hunde; promediar modelos que cometen los mismos errores no cambia nada; promediar modelos fuertes que fallan en sitios distintos es donde está la ganancia. El bagging crea esa diferencia de forma artificial, remuestreando, y por eso ayuda a los modelos que cambian mucho con sus datos, como los árboles profundos, y apenas a los que no, como k-NN o la regresión lineal. El stacking puede aprender de qué miembros fiarse, a costa de más entrenamiento y un modelo más que validar.

En las competiciones, los ensembles de varios modelos de boosting y redes suelen ganar por márgenes pequeños. En la práctica, cada miembro es un modelo más que entrenar, vigilar y explicar, y una ganancia de unos cientos de dólares de error tiene que compensarlo.